# 05 - Workflow agentique RetainFlow

Ce notebook execute les agents RetainFlow etape par etape. La logique vit dans `src/retainflow`; le notebook sert uniquement a orchestrer, afficher les resultats et comprendre le raisonnement.

## 1. Imports

In [ ]:
from retainflow.agents import (
    CustomerProfileAgent,
    DataVisualizationAgent,
    EmailDraftingAgent,
    ExplainabilityAgent,
    KPIAgent,
    RetentionAdvisorAgent,
    SQLAgent,
    SupervisorAgent,
)
from retainflow.config import load_churn_model_config
from retainflow.logging import get_logger

logger = get_logger("retainflow.notebooks.agentic_workflow")

## 2. Configuration

In [ ]:
config = load_churn_model_config("config/churn_model.yml")
config

## 3. Initialisation des agents

In [ ]:
sql_agent = SQLAgent(config)
customer_profile_agent = CustomerProfileAgent(config)
kpi_agent = KPIAgent(config)
retention_agent = RetentionAdvisorAgent(config)
explainability_agent = ExplainabilityAgent(config)
visualization_agent = DataVisualizationAgent()
email_agent = EmailDraftingAgent()

supervisor = SupervisorAgent(
    config=config,
    sql_agent=sql_agent,
    customer_profile_agent=customer_profile_agent,
    kpi_agent=kpi_agent,
    retention_agent=retention_agent,
    explainability_agent=explainability_agent,
    visualization_agent=visualization_agent,
    email_agent=email_agent,
)
supervisor

## 4. Agent SQL - recuperer les donnees

In [ ]:
question_sql = "Quels sont les clients a contacter en urgence ?"
sql_response = sql_agent.answer(question_sql, limit=5)
logger.info(sql_response.answer)
sql_response.data

## 5. Requete SQL source

In [ ]:
print(sql_response.metadata["sql"])

## 6. Agent KPI

In [ ]:
kpi_response = kpi_agent.answer("Montre le volume de clients prioritaires par region")
logger.info(kpi_response.answer)
kpi_response.data

## 7. Agent visualisation - Plotly Express

In [ ]:
visual_response = visualization_agent.answer(
    "Visualise les clients prioritaires par region",
    kpi_response.data,
)
logger.info(visual_response.answer)
visual_response.data.show()

## 8. Exemple complet superviseur - SQL puis visuel

In [ ]:
supervisor_visual_response = supervisor.answer(
    "Visualise le taux de clients contactes cette semaine par agence",
    limit=20,
)
logger.info(supervisor_visual_response.answer)
supervisor_visual_response.data.show()

## 9. Agent conseil retention

In [ ]:
advisor_response = retention_agent.top_recommendations(limit=5)
logger.info(advisor_response.answer)
advisor_response.data

## 10. Agent explicabilite SHAP

In [ ]:
explainability_response = explainability_agent.global_drivers(top_n=10)
logger.info(explainability_response.answer)
explainability_response.data

## 11. Agent redaction email

In [ ]:
email_response = email_agent.draft_first(advisor_response.data)
logger.info(email_response.answer)
email_response.data

## 12. Reponse superviseur sans visuel

In [ ]:
supervisor_response = supervisor.answer(
    "Quels sont les 5 clients que je dois contacter en urgence et pourquoi ?",
    limit=5,
)
logger.info(supervisor_response.answer)
supervisor_response.data